# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking/scoring problem. The goal is not to label each page
as "declining" or "not declining" in isolation, but to put all pages
in order from most-urgent-to-review to least-urgent, because a reviewer
only has time to check a handful of pages and needs to know which ones
to look at first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is is_declining_label, built from the trend_direction column.
This is a proxy, not a clean observed outcome — it's based on the current
90-day window, not on what actually happens to the page afterward. A
stronger version would use a forward-looking label (does the page decline
over the NEXT 30 days), not the current window.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric is Precision@50: of the top 50 pages the system flags, how
many are actually declining. This matters more than overall accuracy
because reviewer time is limited — what matters is whether the top of
the queue is trustworthy, not the whole 30,000-page list.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Shams-Sajid-Rahman/Sajid_FlyRank_AI"
REPO_DIR = "Sajid_FlyRank_AI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df[["content_id", "client_id", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "is_declining_label"]].head(10)

Working directory: /content/Sajid_FlyRank_AI/Sajid_FlyRank_AI


,content_id,client_id,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,content_304f48230142,client_f369cb89fc,20,3803,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,25,15320,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,20,12581,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,22,11751,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,14,19140,44.0,0.13,1
5,content_d4084a4bc775,client_f369cb89fc,20,3970,8.5,0.03,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,1
7,content_a63219c6e95a,client_19581e27de,22,1724,21.2,0.06,0
8,content_5e6c160719bc,client_6208ef0f77,20,32574,46.0,0.09,1
9,content_c27558df2b0c,client_19581e27de,104,1240,4.9,0.16,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule (page not updated in 180+ days AND gets 500+ impressions)
is accurate when it fires — 94% of flagged pages really are declining —
but it only catches 17 out of 30,000 pages. That leaves almost the
entire site unscored. A trained model can weigh many signals together
(staleness, position, CTR, word count) and score every page, catching
far more real problem pages — a random forest got 3x better precision
at the top of the queue than the rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.